# Download Yahoo daily index data

从 Yahoo Finance 下载 2003 年起的日频指数数据，并保存到 `dataset/stock/daily`。

In [16]:
import csv
import datetime as dt
import json
import math
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.parse import quote
from urllib.request import Request, urlopen


In [17]:
DAILY_DIR = Path("dataset/stock/daily")
START_DATE = dt.date(2003, 1, 1)
END_DATE = dt.date.today() + dt.timedelta(days=1)

INDEXES = [
    ("FTSE Straits Times Singapore Daily Historical Results Price Data.csv", "^STI"),
    ("Hang Seng Daily Historical Results Price Data.csv", "^HSI"),
    ("KOSPI Daily Historical Results Price Data (1).csv", "^KS11"),
    ("Nikkei 225 Daily Historical Results Price Data.csv", "^N225"),
    ("S&P 500 Daily Historical Results Price Data.csv", "^GSPC"),
    ("Shanghai Composite Daily Historical Results Price Data.csv", "000001.SS"),
]


In [18]:
def yahoo_chart_url(ticker):
    period1 = int(dt.datetime.combine(START_DATE, dt.time.min, dt.timezone.utc).timestamp())
    period2 = int(dt.datetime.combine(END_DATE, dt.time.min, dt.timezone.utc).timestamp())
    return (
        f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(ticker, safe='')}"
        f"?period1={period1}&period2={period2}&interval=1d&events=history"
        "&includeAdjustedClose=true"
    )


def fetch_chart(ticker):
    req = Request(
        yahoo_chart_url(ticker),
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125 Safari/537.36"
            )
        },
    )
    with urlopen(req, timeout=45) as response:
        payload = json.loads(response.read().decode("utf-8"))

    chart = payload.get("chart", {})
    if chart.get("error"):
        raise RuntimeError(chart["error"])
    result = chart.get("result") or []
    if not result:
        raise RuntimeError(f"No Yahoo chart result for {ticker}")
    return result[0]


In [19]:
def clean_number(value):
    if value is None:
        return None
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return float(value)


def format_price(value):
    return "" if value is None else f"{value:,.2f}"


def format_volume(value):
    if value is None:
        return ""
    value = float(value)
    for suffix, divisor in [("B", 1_000_000_000), ("M", 1_000_000), ("K", 1_000)]:
        if abs(value) >= divisor:
            return f"{value / divisor:.2f}".rstrip("0").rstrip(".") + suffix
    return str(int(value))


In [20]:
def build_rows(chart):
    timestamps = chart.get("timestamp") or []
    quote_data = (chart.get("indicators", {}).get("quote") or [{}])[0]
    opens = quote_data.get("open") or []
    highs = quote_data.get("high") or []
    lows = quote_data.get("low") or []
    closes = quote_data.get("close") or []
    volumes = quote_data.get("volume") or []

    chronological = []
    for i, ts in enumerate(timestamps):
        close = clean_number(closes[i] if i < len(closes) else None)
        open_ = clean_number(opens[i] if i < len(opens) else None)
        high = clean_number(highs[i] if i < len(highs) else None)
        low = clean_number(lows[i] if i < len(lows) else None)
        if close is None or open_ is None or high is None or low is None:
            continue
        chronological.append(
            {
                "date": dt.datetime.fromtimestamp(ts, dt.timezone.utc).date(),
                "close": close,
                "open": open_,
                "high": high,
                "low": low,
                "volume": clean_number(volumes[i] if i < len(volumes) else None),
            }
        )

    rows = []
    previous_close = None
    for item in chronological:
        change_text = ""
        if previous_close:
            change_text = f"{((item['close'] / previous_close) - 1) * 100:.2f}%"
        rows.append(
            {
                "Date": item["date"].strftime("%d/%m/%Y"),
                "Price": format_price(item["close"]),
                "Open": format_price(item["open"]),
                "High": format_price(item["high"]),
                "Low": format_price(item["low"]),
                "Vol.": format_volume(item["volume"]),
                "Change %": change_text,
            }
        )
        previous_close = item["close"]

    rows.reverse()
    return rows


In [21]:
def write_csv(path, rows):
    if not rows:
        raise RuntimeError(f"No rows to write for {path.name}")
    fieldnames = ["Date", "Price", "Open", "High", "Low", "Vol.", "Change %"]
    with NamedTemporaryFile("w", newline="", encoding="utf-8", delete=False, dir=path.parent) as tmp:
        writer = csv.DictWriter(tmp, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        writer.writerows(rows)
        tmp_path = Path(tmp.name)
    tmp_path.replace(path)


In [23]:
DAILY_DIR.mkdir(parents=True, exist_ok=True)

for filename, ticker in INDEXES:
    output_path = DAILY_DIR / filename
    rows = build_rows(fetch_chart(ticker))
    write_csv(output_path, rows)
    print(
        f"{filename}: {ticker}, rows={len(rows)}, "
        f"latest={rows[0]['Date']}, earliest={rows[-1]['Date']}"
    )


FTSE Straits Times Singapore Daily Historical Results Price Data.csv: ^STI, rows=5867, latest=12/06/2026, earliest=02/01/2003
Hang Seng Daily Historical Results Price Data.csv: ^HSI, rows=5777, latest=12/06/2026, earliest=02/01/2003
KOSPI Daily Historical Results Price Data (1).csv: ^KS11, rows=5780, latest=12/06/2026, earliest=02/01/2003
Nikkei 225 Daily Historical Results Price Data.csv: ^N225, rows=5736, latest=12/06/2026, earliest=06/01/2003
S&P 500 Daily Historical Results Price Data.csv: ^GSPC, rows=5900, latest=15/06/2026, earliest=02/01/2003
Shanghai Composite Daily Historical Results Price Data.csv: 000001.SS, rows=5685, latest=12/06/2026, earliest=02/01/2003
